# Presentacion: Analisis integral Chile (version slides)

Enfoque visual para presentacion:
- Filtro LACNIC/IXP/vecinos
- Cobertura de enriquecimiento
- Interconexion estructural
- Cierre de avances


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from IPython.display import display, Markdown

DATA_DIR = Path('../data/csv') if Path('../data/csv').exists() else Path('data/csv')


def load_dataset(name):
    n = pd.read_csv(DATA_DIR / name / 'nodes.csv')
    e = pd.read_csv(DATA_DIR / name / 'edges.csv')
    n['name'] = n['name'].fillna('')
    n['degree'] = n['in_degree'] + n['out_degree']
    return n, e

merged_nodes, merged_edges = load_dataset('merged')
bgp_nodes, bgp_edges = load_dataset('bgp')
ripe_nodes, ripe_edges = load_dataset('ripe_atlas')

asn_b = set(bgp_nodes['asn'].astype(int))
asn_r = set(ripe_nodes['asn'].astype(int))

chile_explicit = set(merged_nodes.loc[merged_nodes['name'].str.contains(r'\bchile\b', case=False, regex=True), 'asn'].astype(int))
chile_manual = {27986, 6568, 27925, 27651, 22047, 52341, 18822, 14117, 10834, 20015, 6471, 6429, 14259, 27678, 20191, 23140, 64112}
asns_seed_ampliado = chile_explicit | chile_manual
asns_lacnic_proxy = set(chile_explicit)

PEERINGDB_COLS = [c for c in merged_nodes.columns if c.startswith('info_') or c.startswith('policy_') or c in ['org_id', 'ix_count', 'fac_count', 'policy_url']]
def has_peeringdb_attrs(df):
    tmp = df[PEERINGDB_COLS].copy()
    for c in tmp.columns:
        if tmp[c].dtype == object:
            tmp[c] = tmp[c].fillna('').astype(str).str.strip().replace('', np.nan)
    return tmp.notna().any(axis=1)

merged_nodes['has_peeringdb_attrs'] = has_peeringdb_attrs(merged_nodes)
print('Merged nodos/aristas:', len(merged_nodes), len(merged_edges))

Merged nodos/aristas: 9839 29649


## Slide 1: Filtro Chile (LACNIC / IXP / vecinos)

In [2]:
ixp_chile_asns = set(
    merged_nodes.loc[
        (merged_nodes['asn'].isin(asns_seed_ampliado)) & (merged_nodes['ix_count'].fillna(0) > 0),
        'asn'
    ].astype(int)
)
agregados_por_ixp = ixp_chile_asns - asns_lacnic_proxy

id2asn = dict(zip(merged_nodes['node_id'], merged_nodes['asn']))
neighbors = set()
for s, d in merged_edges[['src_id', 'dst_id']].itertuples(index=False):
    a = int(id2asn[s]); b = int(id2asn[d])
    if a in asns_seed_ampliado and b not in asns_seed_ampliado:
        neighbors.add(b)
    if b in asns_seed_ampliado and a not in asns_seed_ampliado:
        neighbors.add(a)

groups = [
    ('ASNs delegados (proxy LACNIC)', asns_lacnic_proxy),
    ('ASNs conectados a IXP chileno', ixp_chile_asns),
    ('ASNs agregados por regla IXP', agregados_por_ixp),
    ('ASNs vecinos topologicos', neighbors),
]
rows = []
for gname, gset in groups:
    rows.append({'grupo': gname, 'categoria': 'En LACNIC proxy', 'cantidad': len(gset & asns_lacnic_proxy)})
    rows.append({'grupo': gname, 'categoria': 'Fuera LACNIC proxy', 'cantidad': len(gset - asns_lacnic_proxy)})

filtro_df = pd.DataFrame(rows)
fig = px.bar(filtro_df, x='grupo', y='cantidad', color='categoria', barmode='stack', text='cantidad',
             title='Desglose del filtrado de ASNs para Chile')
fig.update_layout(colorway=['#2ca02c', '#ff7f0e'])
fig.update_traces(textposition='outside')
fig.show()

## Slide 2: Cobertura de atributos PeeringDB

In [3]:
total_as = len(merged_nodes)
con_attrs = int(merged_nodes['has_peeringdb_attrs'].sum())
sin_attrs = total_as - con_attrs

donut = pd.DataFrame({'grupo': ['Con atributos', 'Sin atributos'], 'cantidad': [con_attrs, sin_attrs]})
fig = px.pie(donut, names='grupo', values='cantidad', hole=0.5,
             title=f'Cobertura de enriquecimiento (PeeringDB proxy): {100*con_attrs/total_as:.1f}%')
fig.show()

## Slide 3: Backbone de interconexion

In [4]:
edge_tmp = merged_edges.copy()
edge_tmp['src_asn'] = edge_tmp['src_id'].map(id2asn)
edge_tmp['dst_asn'] = edge_tmp['dst_id'].map(id2asn)
edge_seed = edge_tmp[(edge_tmp['src_asn'].isin(asns_seed_ampliado)) | (edge_tmp['dst_asn'].isin(asns_seed_ampliado))].copy()
backbone_edges = edge_seed.sort_values('weight', ascending=False).head(80)

H = nx.Graph()
for r in backbone_edges.itertuples(index=False):
    H.add_edge(int(r.src_asn), int(r.dst_asn), weight=float(r.weight))

pos = nx.spring_layout(H, seed=7, k=0.45)

ex, ey = [], []
for u, v in H.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    ex += [x0, x1, None]; ey += [y0, y1, None]

edge_trace = go.Scatter(x=ex, y=ey, mode='lines', hoverinfo='none', line=dict(width=0.7, color='#999'))

nx_, ny_, txt, siz, col = [], [], [], [], []
for n in H.nodes():
    x, y = pos[n]
    nx_.append(x); ny_.append(y)
    d = H.degree[n]
    siz.append(6 + 2.2 * np.log1p(d))
    if n in asns_lacnic_proxy:
        c = '#2ca02c'; t = 'LACNIC proxy'
    elif n in asns_seed_ampliado:
        c = '#1f77b4'; t = 'Seed ampliado'
    else:
        c = '#ff7f0e'; t = 'Vecino agregado'
    col.append(c)
    txt.append(f'AS{n} | {t} | grado_local={d}')

node_trace = go.Scatter(x=nx_, y=ny_, mode='markers', text=txt, hoverinfo='text', marker=dict(size=siz, color=col, line=dict(width=0.5, color='white')))

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(title='Backbone de interconexion (top 80 enlaces)', showlegend=False,
                  xaxis=dict(showgrid=False, zeroline=False, visible=False),
                  yaxis=dict(showgrid=False, zeroline=False, visible=False))
fig.show()

## Slide 4: Mapa de calor de interconexion

In [5]:
top_nodes = [n for n, _ in sorted(H.degree, key=lambda x: x[1], reverse=True)[:20]]
adj = np.zeros((len(top_nodes), len(top_nodes)))
for i, u in enumerate(top_nodes):
    for j, v in enumerate(top_nodes):
        if H.has_edge(u, v):
            adj[i, j] = H[u][v].get('weight', 1.0)

labels = [f'AS{n}' for n in top_nodes]
fig = px.imshow(adj, x=labels, y=labels, color_continuous_scale='YlOrRd',
                title='Interconexion ponderada entre ASNs mas conectados')
fig.update_xaxes(side='bottom')
fig.show()

## Slide 5: Interconexion interna vs frontera

In [6]:
edge_seed['categoria'] = np.where(
    edge_seed['src_asn'].isin(asns_seed_ampliado) & edge_seed['dst_asn'].isin(asns_seed_ampliado),
    'Interno seed chileno',
    'Frontera seed chileno'
)

fig = px.box(edge_seed, x='categoria', y='weight', points='outliers', log_y=True,
             title='Peso de interconexion: interno vs frontera')
fig.show()

display(edge_seed.groupby('categoria')['weight'].agg(['count', 'mean', 'median', 'max']).reset_index())

,categoria,count,mean,median,max
0,Frontera seed chileno,211,609.838863,30.0,21633
1,Interno seed chileno,23,2970.391304,20.0,25691


## Slide 6: Resumen ejecutivo

In [7]:
resumen = pd.DataFrame([
    {'componente': 'Filtro Chile', 'estado': 'OK', 'resultado': f'seed={len(asns_seed_ampliado)} | vecinos={len(neighbors)}'},
    {'componente': 'PeeringDB proxy', 'estado': 'OK', 'resultado': f'cobertura={100*con_attrs/total_as:.1f}%'},
    {'componente': 'Backbone', 'estado': 'OK', 'resultado': f'nodos={len(H.nodes())} | enlaces={len(H.edges())}'},
    {'componente': 'Interconexion', 'estado': 'OK', 'resultado': 'heatmap + caja interno/frontera listos'},
])

display(resumen)

mensaje = [
    '- La presentacion muestra pipeline de filtro, enriquecimiento y estructura en formato visual.',
    '- Separamos claramente ASNs LACNIC proxy, IXP y vecinos agregados.',
    '- Queda lista para exposicion con foco en interconexion y hallazgos.'
]
display(Markdown("\\n".join(mensaje)))

,componente,estado,resultado
0,Filtro Chile,OK,seed=34 | vecinos=143
1,PeeringDB proxy,OK,cobertura=70.7%
2,Backbone,OK,nodos=76 | enlaces=80
3,Interconexion,OK,heatmap + caja interno/frontera listos


- La presentacion muestra pipeline de filtro, enriquecimiento y estructura en formato visual.\n- Separamos claramente ASNs LACNIC proxy, IXP y vecinos agregados.\n- Queda lista para exposicion con foco en interconexion y hallazgos.